In [50]:
import os
import urllib.request

# ==========================================
# 1. DATASET SETUP
# ==========================================
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filename = "input.txt"

# Download the tiny shakespeare dataset if it doesn't exist
if not os.path.exists(filename):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete.")
else:
    print("Dataset already exists locally.")

# Read the text in to extract characters
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# Get all unique characters and sort them
chars = sorted(list(set(text)))
vocab_size = len(chars)


# ==========================================
# 2. TOKENIZER DICTIONARY MAPPINGS
# ==========================================
stoi = {}
for ind, ch in enumerate(chars):
    stoi[ch] = ind

itos = {}
for ind, ch in enumerate(chars):
    itos[ind] = ch


# ==========================================
# 3. TOKENIZER FUNCTIONS (LOOP IMPLEMENTATION)
# ==========================================
def encode(string: str) -> list[int]:
    arr = []
    for loop in range(len(string)):
        arr.append(stoi[string[loop]])
    return arr

def decode(arr: list[int]) -> str:
    ans = ''
    for loop in range(len(arr)):
        ans = ans + itos[arr[loop]]
    return ans


# ==========================================
# 4. TESTING THE TOKENIZER
# ==========================================
print("\n--- Tokenizer Test ---")
test_name = "prasanna"
encoded_ids = encode(test_name)
decoded_back = decode(encoded_ids)

print(f"Original: {test_name}")
print(f"Encoded IDs: {encoded_ids}")
print(f"Decoded Back: {decoded_back}")
print("-" * 30)



Dataset already exists locally.

--- Tokenizer Test ---
Original: prasanna
Encoded IDs: [54, 56, 39, 57, 39, 52, 52, 39]
Decoded Back: prasanna
------------------------------


In [51]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)  # one giant encoded vector 

split_size  =  int(0.9 * len(data))     # split size 
train_data  = data[:split_size]         # training data
val_data = data[split_size:]            # valodation data


In [52]:
context_size = 8 
train_data[:context_size+1]

x = train_data[:context_size]
y = train_data[1:context_size + 1 ]

for t in range(context_size):
    context = x[:t+1]
    target = y[t]
    print(context,target)



tensor([18]) tensor(47)
tensor([18, 47]) tensor(56)
tensor([18, 47, 56]) tensor(57)
tensor([18, 47, 56, 57]) tensor(58)
tensor([18, 47, 56, 57, 58]) tensor(1)
tensor([18, 47, 56, 57, 58,  1]) tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) tensor(58)


In [53]:
torch.manual_seed(1337)
batch_size = 4 

# device selection: use CUDA if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def get_batch(split : str):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_size,(batch_size,))
    x = torch.stack([data[i:i+context_size] for i in ix])
    y = torch.stack([data[i+1: i + context_size + 1] for i in ix])
    return x.to(device) , y.to(device)

xb , yb = get_batch('train')

print("inputs")
print(xb.shape )
print('xb device:', xb.device)
print(xb)

print('outputs')
print(yb.shape)
print('yb device:', yb.device)
print(yb)

    
print("-------------------------------------------------------")


for b in range(batch_size):
    for t in range(context_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"input = {context.tolist()} , the target is {target}")


Using device: cuda
inputs
torch.Size([4, 8])
xb device: cuda:0
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
outputs
torch.Size([4, 8])
yb device: cuda:0
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')
-------------------------------------------------------
input = [24] , the target is 43
input = [24, 43] , the target is 58
input = [24, 43, 58] , the target is 5
input = [24, 43, 58, 5] , the target is 57
input = [24, 43, 58, 5, 57] , the target is 1
input = [24, 43, 58, 5, 57, 1] , the target is 46
input = [24, 43, 58, 5, 57, 1, 46] , the target is 43
input = [24, 43, 58, 5, 57, 1, 46, 43] , the target is 39
input = [44] , the target is 53
input = [44, 53] , the target is 56
input = [44, 53, 56] , the target is 1

In [ ]:
import torch.nn as nn 
from torch.nn import functional as F
torch.manual_seed(1337)


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            return logits
        B, T, C = logits.shape
        logits = logits.view(B * T, C)
        targets = targets.view(B * T)
        loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) of current context tokens
        for _ in range(max_new_tokens):
            logits = self.token_embedding_table(idx)  # (B, T, C)
            logits_last = logits[:, -1, :]  # (B, C)
            probs = F.softmax(logits_last, dim=-1)  # (B, C)
            next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_token], dim=1)  # append
        return idx



m = BigramLanguageModel(vocab_size)
# move model to selected device
m = m.to(device)

# example forward (with loss) if desired
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


torch.Size([32, 65])
tensor(4.8786, device='cuda:0', grad_fn=<NllLossBackward0>)


In [57]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)
batch_size = 64
for iter in range(100000):
    xb , yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
    
print(loss.item())


2.424154281616211


In [ ]:
idx  = torch.zeros((1,1) , dtype = torch.long, device=device)
print(decode(m.generate(idx, max_new_tokens = 300)[0].tolist()))


AttributeError: 'BigramLanguageModel' object has no attribute 'generate'